# Digital Finance Cleaning
Remove the Correction entries from the data, as these are not independent articles.  

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("Data/digital_finance_metadata.csv")
print(f"原始数据: {len(df)} 行")

# ============ 清洗①：删除 erratum 类文章 ============
pattern = r"^(correction|erratum|errata|publisher.?s note|retraction|editorial expression of concern)\b"
mask_title = df["title"].str.match(pattern, case=False, na=False)
print(f"\n将删除 erratum 类 {mask_title.sum()} 行：")
print(df.loc[mask_title, ["title", "year"]].to_string())

# ============ 清洗②：删除 abstract 为空的文章 ============
# 视为"空"的三种情况：NaN、"nan"/"none"字符串、清洗后为空的字符串
abs_raw = df["abstract"].astype(str).str.strip()
mask_abs_empty = (
    df["abstract"].isna()                          # 真 NaN
    | abs_raw.str.lower().isin(["nan", "none", "null", ""])   # 空串或占位符
)
print(f"\n将删除 abstract 为空的 {mask_abs_empty.sum()} 行：")
print(df.loc[mask_abs_empty, ["title", "year"]].to_string())

原始数据: 216 行

将删除 erratum 类 10 行：
                                                                                                                  title  year
41                                                  Correction: Interest rate models in decentralised lending protocols  2026
59                                                           Correction: Crypto-assets and the risk of money laundering  2026
61                              Correction: Digital business world and ethical dilemmas: a systematic literature review  2025
71                                                        Correction: Regime switching forecasting for cryptocurrencies  2025
72      Correction: Understanding temporal dynamics of jumps in cryptocurrency markets: evidence from tick-by-tick data  2025
103                                                        Correction: Does investor sentiment drive the Bitcoin price?  2025
111   Correction: Predicting the reaction of financial markets to Federal Open Market

In [3]:
# ============ 确认无误后，执行删除 ============
df_clean = df[~mask_title & ~mask_abs_empty].copy()

df_clean.to_csv("Data/digital_finance_aftercleaning.csv", index=False, encoding="utf-8-sig")
print(f"\n清洗后: {len(df_clean)} 行"
      f"（删 erratum {mask_title.sum()} + 删空摘要 {mask_abs_empty.sum()}，"
      f"合计 {mask_title.sum() + mask_abs_empty.sum()}），"
      f"已保存到 Data/digital_finance_aftercleaning.csv")


清洗后: 194 行（删 erratum 10 + 删空摘要 22，合计 32），已保存到 Data/digital_finance_aftercleaning.csv
